In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml down minio

In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml up -d minio


#### Delete Existing Delta Table

In [ ]:
%%bash

## Delete Existing Delta Table
aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/ --recursive

## Delete Existing Delta Table
aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/spark/ --recursive


In [1]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [2]:
%sql spark

#### Create Database

In [ ]:
%%sql

ALTER DATABASE default SET LOCATION 's3://warehouse/default/spark'

Running query in 'SparkSession'

26/05/21 08:14:31 ERROR Schema: Failed initialising database.
Unable to open a test connection to the given database. JDBC url = jdbc:derby:;databaseName=/apps/sandbox/metastore;create=true, username = APP. Terminating connection pool (set lazyInit to true if you expect to start your database after your app). Original Exception: ------
java.sql.SQLException: Failed to start database '/apps/sandbox/metastore' with class loader jdk.internal.loader.ClassLoaders$AppClassLoader@5ffd2b27, see the next exception for details.
	at org.apache.derby.impl.jdbc.SQLExceptionFactory.getSQLException(Unknown Source)
	at org.apache.derby.impl.jdbc.SQLExceptionFactory.getSQLException(Unknown Source)
	at org.apache.derby.impl.jdbc.Util.seeNextException(Unknown Source)
	at org.apache.derby.impl.jdbc.EmbedConnection.bootDatabase(Unknown Source)
	at org.apache.derby.impl.jdbc.EmbedConnection.<init>(Unknown Source)
	at org.apache.derby.jdbc.InternalDriver$1.run(Unknown Source)
	at org.apache.derby.jdbc.Intern

KeyboardInterrupt: 

26/05/21 08:14:43 ERROR Schema: Failed initialising database.
Unable to open a test connection to the given database. JDBC url = jdbc:derby:;databaseName=/apps/sandbox/metastore;create=true, username = APP. Terminating connection pool (set lazyInit to true if you expect to start your database after your app). Original Exception: ------
java.sql.SQLException: Failed to start database '/apps/sandbox/metastore' with class loader jdk.internal.loader.ClassLoaders$AppClassLoader@5ffd2b27, see the next exception for details.
	at org.apache.derby.impl.jdbc.SQLExceptionFactory.getSQLException(Unknown Source)
	at org.apache.derby.impl.jdbc.SQLExceptionFactory.getSQLException(Unknown Source)
	at org.apache.derby.impl.jdbc.Util.seeNextException(Unknown Source)
	at org.apache.derby.impl.jdbc.EmbedConnection.bootDatabase(Unknown Source)
	at org.apache.derby.impl.jdbc.EmbedConnection.<init>(Unknown Source)
	at org.apache.derby.jdbc.InternalDriver$1.run(Unknown Source)
	at org.apache.derby.jdbc.Intern

In [25]:
%%sql

USE spark_catalog.default;

Running query in 'SparkSession'

++
||
++
++

In [26]:
%%sql 

DESCRIBE SCHEMA spark_catalog.default;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2
Catalog Name,spark_catalog
Namespace Name,default
Comment,Default Hive database
Location,s3a://warehouse/default/spark
Owner,public


In [27]:
%%sql 

show tables in spark_catalog.default;

Running query in 'SparkSession'

2 rows affected.

Field 1,Field 2,Field 3
default,people_table,False
default,spark_table,False


#### Spark Tables

In [5]:
%%sql 

--
DROP TABLE IF EXISTS spark_catalog.default.spark_table;

--
CREATE TABLE spark_catalog.default.spark_table (
    id INT,
    firstName STRING,
    middleName STRING,
    lastName STRING,
    gender STRING,
    birthDate TIMESTAMP,
    ssn STRING,
    salary INT
)
--LOCATION 's3a://warehouse/default/spark/spark_table'

Running query in 'SparkSession'

++
||
++
++

In [6]:
%%sql 

show tables in spark_catalog.default;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3
default,spark_table,False


In [7]:
%%sql 

show create table spark_catalog.default.spark_table;

Running query in 'SparkSession'

1 rows affected.

Field 1
"CREATE TABLE default.spark_table ( id INT, firstName STRING, middleName STRING, lastName STRING, gender STRING, birthDate TIMESTAMP, ssn STRING, salary INT)USING textTBLPROPERTIES ( 'transient_lastDdlTime' = '1780063938')"


In [8]:
%%sql 

DESCRIBE TABLE FORMATTED spark_catalog.default.spark_table

Running query in 'SparkSession'

26 rows affected.

Field 1,Field 2,Field 3
id,int,None
firstName,string,None


In [9]:
%%sql 

DESCRIBE HISTORY spark_catalog.default.spark_table;

Running query in 'SparkSession'

RuntimeError: [DELTA_ONLY_OPERATION] DESCRIBE HISTORY is only supported for Delta tables.


#### Delta Tables

In [10]:
%%sql 

show SCHEMAS;

Running query in 'SparkSession'

2 rows affected.

Field 1
default
deltalake


In [4]:
%%sql

DROP SCHEMA IF EXISTS deltalake;

Running query in 'SparkSession'

++
||
++
++

In [5]:
%%sql
-- In Spark SQL, the terms DATABASE and SCHEMA are interchangeable synonyms;
CREATE DATABASE IF NOT EXISTS deltalake
COMMENT 'This is a default database for the deltalake'
LOCATION 's3a://warehouse/default/deltalake';

Running query in 'SparkSession'

++
||
++
++

In [ ]:
%%sql

CREATE SCHEMA IF NOT EXISTS deltalake
COMMENT 'This is a default database for the deltalake'
LOCATION 's3a://warehouse/default/deltalake';


In [6]:
%%sql 

show SCHEMAS;

Running query in 'SparkSession'

2 rows affected.

Field 1
default
deltalake


In [11]:
%%sql 

DESC SCHEMA deltalake;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2
Catalog Name,spark_catalog
Namespace Name,deltalake
Comment,This is a default database for the deltalake
Location,s3a://warehouse/default/deltalake
Owner,brijeshdhaker


In [12]:
%%sql 

DESCRIBE DATABASE deltalake;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2
Catalog Name,spark_catalog
Namespace Name,deltalake
Comment,This is a default database for the deltalake
Location,s3a://warehouse/default/deltalake
Owner,brijeshdhaker


In [ ]:
%%sql

ALTER DATABASE deltalake SET LOCATION 's3a://warehouse/default/deltalake'

In [13]:
%%sql

USE deltalake;

Running query in 'SparkSession'

++
||
++
++

In [15]:
%%sql 

--
DROP TABLE IF EXISTS spark_catalog.deltalake.delta_table;

--
CREATE TABLE IF NOT EXISTS spark_catalog.deltalake.delta_table (
    id INT,
    firstName STRING,
    middleName STRING,
    lastName STRING,
    gender STRING,
    birthDate TIMESTAMP,
    ssn STRING,
    salary INT
)
USING DELTA
--LOCATION 's3a://warehouse/default/deltalake/delta_table'

Running query in 'SparkSession'

++
||
++
++

In [14]:
%%sql 

show tables in spark_catalog.deltalake;

Running query in 'SparkSession'

4 rows affected.

Field 1,Field 2,Field 3
deltalake,lc_ex1,False
deltalake,lc_ex2,False
deltalake,zorder_ex1,False
deltalake,zorder_ex2,False


In [16]:
%%sql 

DESCRIBE TABLE FORMATTED spark_catalog.deltalake.delta_table

Running query in 'SparkSession'

16 rows affected.

Field 1,Field 2,Field 3
id,int,None
firstName,string,None
middleName,string,None
lastName,string,None
gender,string,None
birthDate,timestamp,None
ssn,string,None
salary,int,None
,,
# Detailed Table Information,,


In [17]:
%%sql 

DESCRIBE TABLE EXTENDED spark_catalog.deltalake.delta_table;

Running query in 'SparkSession'

16 rows affected.

Field 1,Field 2,Field 3
id,int,None
firstName,string,None
middleName,string,None
lastName,string,None
gender,string,None
birthDate,timestamp,None
ssn,string,None
salary,int,None
,,
# Detailed Table Information,,


In [18]:
%%sql 

DESCRIBE HISTORY spark_catalog.deltalake.delta_table;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
0,2026-05-29 19:43:55,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,True,{},None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [19]:
%%sql 

DESCRIBE DETAIL spark_catalog.deltalake.delta_table;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
delta,1dd855f3-98f6-4390-8b2d-6df6f25300ed,spark_catalog.deltalake.delta_table,None,s3a://warehouse/default/deltalake/delta_table,2026-05-29 19:43:55.420000,2026-05-29 19:43:55,[],[],0,0,{},1,2,"['appendOnly', 'invariants']"


##### List catalogs in the current/default database

In [20]:
# Get a list of all catalogs
catalogs = spark.catalog.listCatalogs()

# Iterate and print each catalog name
print("Available catalogs:")
for catalog in catalogs:
    # The 'catalog' object has a 'name' attribute
    print(f"* {catalog.name}") 

Available catalogs:
* spark_catalog


In [21]:
# Get the name of the current catalog
current_catalog = spark.catalog.currentCatalog()
print(f"Current catalog name: {current_catalog}")

Current catalog name: spark_catalog


##### List databases in the current/default catalog

In [22]:

# List databases in the current/default catalog
databases = spark.catalog.listDatabases()
for db in databases :
    print(f"- {db}")

- Database(name='default', catalog='spark_catalog', description='Default Hive database', locationUri='s3a://warehouse/default/spark')
- Database(name='deltalake', catalog='spark_catalog', description='This is a default database for the deltalake', locationUri='s3a://warehouse/default/deltalake')


##### List tables in the current/default database

In [23]:

# List tables in the current/default database
tables_list = spark.catalog.listTables()
print(f"Tables in all database")
for table in tables_list:
    print(f" namespace - {table.namespace}  table - {table.name}")

print()
print("-"*50)
print()
# List tables in a specific database
tables_in_db = spark.catalog.listTables("default")
print(f"Tables in default database")

# Iterate and print table names (optional)
for table in tables_in_db:
    print(f" namespace - {table.namespace}  table - {table.name}")

Tables in all database
 namespace - ['deltalake']  table - delta_table
 namespace - ['deltalake']  table - lc_ex1
 namespace - ['deltalake']  table - lc_ex2
 namespace - ['deltalake']  table - zorder_ex1
 namespace - ['deltalake']  table - zorder_ex2

--------------------------------------------------

Tables in default database
 namespace - ['default']  table - spark_table


In [ ]:
# The spark.catalog object provides many more functions for metadata management: 

# Returns a list of columns and their data types.
spark.catalog.listColumns("delta_table")

# Caches the table data in memory for faster access.
spark.catalog.cacheTable("delta_table")

# Removes the table from the cache.
spark.catalog.uncacheTable("delta_table")

# Checks if a table exists.
spark.catalog.tableExists("delta_table")

##### Save the DataFrame as a table in the spark default database

In [24]:
# Create a sample DataFrame
data = [("James", 34), ("Margaret", 28), ("Robert", 45)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)

# Save the DataFrame as a table in the current database
df.write.saveAsTable("spark_catalog.default.people_table")

##### Create an Managed table:

In [ ]:
# Creates an empty managed table with a defined schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])
spark.catalog.createTable("managed_people_table", schema=schema, source="parquet")

##### Create an External table:

In [ ]:
# Writing a DataFrame to a specific location as an external table
df.write \
  .option("path", "s3a://warehouse/default/spark/my_external_parquet_table") \
  .format("parquet") \
  .saveAsTable("my_external_parquet_table")

##### Create an External table:


In [ ]:
# Define a path where data is stored (e.g., in CSV format)
file_path = "s3a://warehouse/default/spark/my_external_parquet_table"

# Create an external table from the data at the given path
# Registers existing CSV data as an external table
spark.catalog.createTable(
    "external_people_table", 
    path="s3a://warehouse/default/spark/my_external_parquet_table", 
    source="parquet", 
    header="true", 
    inferSchema="true"
)



In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", TimestampType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

peoples_df = spark.read.format("csv").option("header", True).schema(schema).load("s3a://datasets/people-data/peoples.csv")
peoples_df.printSchema()

#
display(peoples_df)

In [ ]:
peoples_df.write.mode("append").saveAsTable("spark_catalog.default.spark_table")

In [ ]:
# If you know the table does not already exist, you can call this instead:
peoples_df.write.mode("append").insertInto("spark_catalog.default.spark_table")

##### Save the DataFrame as a table in the deltalake database

In [ ]:
# Create the table if it does not exist. Otherwise, replace the existing table.
peoples_df.writeTo("spark_catalog.deltalake.delta_table").createOrReplace()

In [ ]:
# Or to a named table:
peoples_df.write.format("delta").mode("append").saveAsTable("spark_catalog.deltalake.delta_table")

In [ ]:
# Save as delta table into Minio S3
peoples_df.write.format('delta').mode("append").save('s3a://warehouse/default/deltalake/delta_table')
